In [ ]:
"""
Static Security Benchmark
This script measures how well Bandit classifies vulnerable and patched code pairs from CSV input.
"""

import json
import os
import sys
import re
import time
import shutil
import subprocess
import tempfile
import pandas as pd
from datetime import datetime
from pathlib import Path
from llmScan import *
from sklearn.metrics import classification_report

In [ ]:
# Shared helpers for paired and unpaired benchmark CSVs
def _normalize_benchmark_label(value):
    if pd.isna(value):
        return "unknown"
    text = str(value).strip().lower()
    label_map = {
        "vulnerable": "vulnerable",
        "vuln": "vulnerable",
        "unsafe": "vulnerable",
        "malicious": "vulnerable",
        "safe": "safe",
        "patched": "safe",
        "benign": "safe",
        "non-vulnerable": "safe",
        "nonvulnerable": "safe",
        "not vulnerable": "safe",
        "true": "vulnerable",
        "false": "safe",
        "1": "vulnerable",
        "0": "safe",
    }
    return label_map.get(text, text or "unknown")


def _report_label(label):
    return "patched" if label == "safe" else label


def _detect_benchmark_input_schema(
    df,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
):
    columns = set(df.columns)
    if before_col in columns and after_col in columns:
        return {"mode": "paired", "before_col": before_col, "after_col": after_col}

    code_candidates = [candidate for candidate in [code_col, "code_sample", "code", "sample_code", "source_code", "snippet", "content"] if candidate]
    label_candidates = [candidate for candidate in [label_col, "sample_label", "label", "ground_truth", "target", "class", "vulnerability_label", "status"] if candidate]

    resolved_code = next((candidate for candidate in code_candidates if candidate in columns), None)
    resolved_label = next((candidate for candidate in label_candidates if candidate in columns), None)

    if resolved_code and resolved_label:
        return {"mode": "single", "code_col": resolved_code, "label_col": resolved_label}

    raise ValueError(
        "Could not detect benchmark schema. Expected either paired columns "
        f"({before_col!r}, {after_col!r}) or a single code/label pair such as "
        "('code_sample', 'sample_label')."
    )


def _benchmark_result_mode(df, prefix):
    if f"{prefix}_classification_before" in df.columns and f"{prefix}_classification_after" in df.columns:
        return "paired"
    if f"{prefix}_classification" in df.columns and (
        f"{prefix}_expected_label" in df.columns or "expected_label" in df.columns
    ):
        return "single"
    return None


def _build_benchmark_report_data(df, prefix):
    mode = _benchmark_result_mode(df, prefix)
    if mode == "paired":
        total_rows = len(df)
        y_true = ["vulnerable"] * total_rows + ["patched"] * total_rows
        y_pred_raw = df[f"{prefix}_classification_before"].tolist() + df[f"{prefix}_classification_after"].tolist()
        y_pred = [_report_label(label) for label in y_pred_raw]
        return {"mode": mode, "y_true": y_true, "y_pred": y_pred, "total_predictions": total_rows * 2}
    if mode == "single":
        expected_col = f"{prefix}_expected_label" if f"{prefix}_expected_label" in df.columns else "expected_label"
        y_true = df[expected_col].tolist()
        y_pred_raw = df[f"{prefix}_classification"].tolist()
        y_true_report = [_report_label(label) for label in y_true]
        y_pred_report = [_report_label(label) for label in y_pred_raw]
        return {"mode": mode, "y_true": y_true_report, "y_pred": y_pred_report, "total_predictions": len(df)}
    return None

In [ ]:
# Define benchmark test for one full code snippet using Bandit

def _bandit_valid_issues(issues):
    return [
        issue for issue in issues
        if str(issue.get("issue_confidence", "")).upper() == "HIGH"
        and str(issue.get("issue_severity", "")).upper() in ("MEDIUM", "HIGH")
    ]


_BANDIT_CHECKED = False
_BANDIT_COMMAND = None
_BANDIT_MISSING_LOGGED = False
_BANDIT_FORCE_MODULE = False


def _is_command_available(cmd):
    """Check whether the first executable in a command list is still valid."""
    if not cmd:
        return False
    exe = cmd[0]
    if os.path.isabs(exe):
        return os.path.exists(exe) and os.access(exe, os.X_OK)
    return shutil.which(exe) is not None


def _get_bandit_command():
    """Resolve Bandit command once: prefer binary, fallback to python -m bandit."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    if _BANDIT_CHECKED:
        # Refresh stale cached paths in long-lived notebook kernels.
        if _is_command_available(_BANDIT_COMMAND):
            return _BANDIT_COMMAND
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None

    _BANDIT_CHECKED = True

    if not _BANDIT_FORCE_MODULE:
        bandit_bin = shutil.which("bandit")
        if bandit_bin:
            _BANDIT_COMMAND = [bandit_bin]
            return _BANDIT_COMMAND

    try:
        probe = subprocess.run(
            [sys.executable, "-m", "bandit", "--version"],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        if probe.returncode == 0:
            _BANDIT_COMMAND = [sys.executable, "-m", "bandit"]
            return _BANDIT_COMMAND
    except Exception:
        pass

    _BANDIT_COMMAND = None
    return _BANDIT_COMMAND


def _run_bandit(command):
    """Run Bandit and recover from broken launcher scripts once."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    try:
        return subprocess.run(
            command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
    except FileNotFoundError:
        # Common case: launcher exists but shebang points to missing interpreter.
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None
        _BANDIT_FORCE_MODULE = True
        refreshed = _get_bandit_command()
        if not refreshed:
            raise
        retry_command = refreshed + command[len(command) - 5 :]
        return subprocess.run(
            retry_command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )


def test_vulnerabilities_benchmark(python_code_text):
    """
    Analyze a full Python code snippet for security vulnerabilities using Bandit.

    Returns:
        (success(bool), is_vulnerable(bool|None), tool_name(str), scan_time(float))
    """
    start_time = time.time()
    tool_name = "Bandit"
    temp_path = None

    try:
        bandit_base = _get_bandit_command()
        if not bandit_base:
            global _BANDIT_MISSING_LOGGED
            if not _BANDIT_MISSING_LOGGED:
                print(
                    "Bandit is not available in this environment. "
                    "Install with: pip install bandit (or pip install -r requirements.txt)."
                )
                _BANDIT_MISSING_LOGGED = True
            scan_time = time.time() - start_time
            return False, None, tool_name, scan_time

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp_file:
            tmp_file.write(str(python_code_text or ""))
            temp_path = tmp_file.name

        command = bandit_base + ["-r", temp_path, "-ll", "-iii", "--format", "json"]
        proc = _run_bandit(command)

        global last_bandit_output
        last_bandit_output = (proc.stdout or "") + "\n" + (proc.stderr or "")

        data = json.loads(proc.stdout or "{}")
        all_issues = data.get("results", [])
        valid_issues = _bandit_valid_issues(all_issues)

        is_vulnerable = len(valid_issues) > 0
        scan_time = time.time() - start_time
        return True, is_vulnerable, tool_name, scan_time

    except Exception as e:
        last_bandit_output = locals().get("last_bandit_output", str(e))
        print(f"Bandit run failed: {e}\nLast output:\n{last_bandit_output[:1000]}")
        scan_time = time.time() - start_time
        return False, None, tool_name, scan_time

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)


def label_from_prediction(success, is_vulnerable):
    if not success or is_vulnerable is None:
        return "unknown"
    return "vulnerable" if is_vulnerable else "safe"


last_bandit_output = ""

print("test_vulnerabilities_benchmark function defined successfully!")

In [ ]:
# Define CSV-driven benchmark runner
def run_benchmark_from_csv(
    csv_path,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
    verbose=True,
):
    """
    Run Bandit benchmark from a CSV in either paired or single-sample form.

    Paired input: one row contains vulnerable and patched snippets.
    Single-sample input: one row contains one code sample plus its label.
    """
    df = pd.read_csv(csv_path)

    schema = _detect_benchmark_input_schema(
        df,
        before_col=before_col,
        after_col=after_col,
        code_col=code_col,
        label_col=label_col,
    )

    output_df = df.copy()
    output_df["benchmark_input_mode"] = schema["mode"]
    model_names = []

    if schema["mode"] == "paired":
        before_predictions = []
        after_predictions = []
        before_correct = []
        after_correct = []
        before_success = []
        after_success = []
        before_completed = []
        after_completed = []
        before_times = []
        after_times = []

        for idx, row in output_df.iterrows():
            before_code = row.get(schema["before_col"], "")
            after_code = row.get(schema["after_col"], "")

            b_completed, b_pred, b_model, b_time = test_vulnerabilities_benchmark(before_code)
            a_completed, a_pred, a_model, a_time = test_vulnerabilities_benchmark(after_code)

            b_label = label_from_prediction(b_completed, b_pred)
            a_label = label_from_prediction(a_completed, a_pred)

            expected_before = "vulnerable"
            expected_after = "safe"
            b_is_correct = b_label == expected_before
            a_is_correct = a_label == expected_after

            before_predictions.append(b_label)
            after_predictions.append(a_label)
            before_correct.append(b_is_correct)
            after_correct.append(a_is_correct)
            before_success.append(b_is_correct)
            after_success.append(a_is_correct)
            before_completed.append(b_completed)
            after_completed.append(a_completed)
            before_times.append(round(b_time, 3))
            after_times.append(round(a_time, 3))
            model_names.append(a_model or b_model or "unknown")

            if verbose:
                print(
                    f"Row {idx + 1}: before={b_label} (expected vulnerable), "
                    f"after={a_label} (expected safe)"
                )

        output_df["expected_before"] = "vulnerable"
        output_df["expected_after"] = "safe"
        output_df["bandit_classification_before"] = before_predictions
        output_df["bandit_classification_after"] = after_predictions
        output_df["bandit_correct_before"] = before_correct
        output_df["bandit_correct_after"] = after_correct
        output_df["bandit_success_before"] = before_success
        output_df["bandit_success_after"] = after_success
        output_df["bandit_completed_before"] = before_completed
        output_df["bandit_completed_after"] = after_completed
        output_df["bandit_scan_time_before_seconds"] = before_times
        output_df["bandit_scan_time_after_seconds"] = after_times
        output_df["bandit_pair_correct"] = output_df["bandit_correct_before"] & output_df["bandit_correct_after"]
    else:
        code_column = schema["code_col"]
        label_column = schema["label_col"]
        predictions = []
        correct = []
        success = []
        completed = []
        scan_times = []
        expected_labels = []

        for idx, row in output_df.iterrows():
            code_text = row.get(code_column, "")
            expected_label = _normalize_benchmark_label(row.get(label_column, ""))

            row_completed, row_pred, row_model, row_time = test_vulnerabilities_benchmark(code_text)
            row_label = label_from_prediction(row_completed, row_pred)

            is_correct = row_label == expected_label

            expected_labels.append(expected_label)
            predictions.append(row_label)
            correct.append(is_correct)
            success.append(is_correct)
            completed.append(row_completed)
            scan_times.append(round(row_time, 3))
            model_names.append(row_model or "unknown")

            if verbose:
                print(
                    f"Row {idx + 1}: expected={expected_label} | "
                    f"predicted={row_label} | completed={row_completed}"
                )

        output_df["expected_label"] = expected_labels
        output_df["bandit_expected_label"] = expected_labels
        output_df["bandit_classification"] = predictions
        output_df["bandit_correct"] = correct
        output_df["bandit_success"] = success
        output_df["bandit_completed"] = completed
        output_df["bandit_scan_time_seconds"] = scan_times
        output_df["bandit_code_column"] = code_column
        output_df["bandit_label_column"] = label_column

    output_df["bandit_model"] = model_names

    return output_df


print("run_benchmark_from_csv function defined successfully!")

In [ ]:
# Run the CSV benchmark
input_csv_path = "splitData2.csv"

print(f"Starting Bandit CSV Security Benchmark on {input_csv_path}...")

bandit_cmd = _get_bandit_command()
if not bandit_cmd:
    raise RuntimeError(
        "Bandit is not installed in the current environment. "
        "Install with: pip install bandit (or pip install -r requirements.txt)."
    )

df_results = run_benchmark_from_csv(input_csv_path, verbose=True)

model_tag = "Bandit"
date_stamp = datetime.now().strftime("%Y%m%d")
output_file = f"benchmark{model_tag}{date_stamp}.csv"
df_results.to_csv(output_file, index=False)

print(f"Benchmark completed for {len(df_results)} rows.")
print(f"Results saved to: {output_file}")

In [ ]:
# Calculate and display comprehensive metrics
from sklearn.metrics import classification_report, roc_curve, auc
import matplotlib.pyplot as plt
print("=" * 60)
print("BANDIT CSV BENCHMARK RESULTS")
print("=" * 60)

total_rows = len(df_results)
print(f"Total rows/pairs analyzed: {total_rows}")

if any(c.startswith("bandit_") for c in df_results.columns):
    metric_prefix = "bandit"
else:
    metric_prefix = "llm"

report_data = _build_benchmark_report_data(df_results, metric_prefix)
if report_data is None:
    print("No supported benchmark output columns found.")
else:
    y_true = report_data["y_true"]
    y_pred = report_data["y_pred"]
    total_predictions = report_data["total_predictions"]

    if report_data["mode"] == "paired":
        failed_before = int((df_results[f"{metric_prefix}_completed_before"] != True).sum()) if total_rows else 0
        failed_after = int((df_results[f"{metric_prefix}_completed_after"] != True).sum()) if total_rows else 0
        tests_failed = failed_before + failed_after
        print(f"Tests failed: {tests_failed}/{total_predictions}")
    else:
        failed = int((df_results[f"{metric_prefix}_completed"] != True).sum()) if f"{metric_prefix}_completed" in df_results.columns and total_rows else 0
        print(f"Tests failed: {failed}/{total_predictions}")

    print("\n" + "-" * 60)
    print("Classification Report (total predictions)")
    print("-" * 60)
    print(classification_report(y_true, y_pred, labels=["vulnerable", "patched"], target_names=["vulnerable", "patched"], zero_division=0, digits=3))

    pred_series = pd.Series(y_pred, name="Predicted")
    pred_series = pred_series.where(pred_series.isin(["vulnerable", "patched"]), "unknown")
    actual_series = pd.Series(y_true, name="Actual")

    confusion_matrix = pd.crosstab(actual_series, pred_series, dropna=False)
    display_cols = ["vulnerable", "patched", "unknown"]
    present_cols = [c for c in display_cols if c in confusion_matrix.columns]
    confusion_matrix = confusion_matrix.reindex(index=["vulnerable", "patched"], columns=present_cols, fill_value=0)
    confusion_matrix.index = [f"Actual: {idx}" for idx in confusion_matrix.index]
    confusion_matrix.columns = [f"Predicted: {col}" for col in confusion_matrix.columns]

    print("-" * 60)
    print("Confusion Matrix (total predictions)")
    print("-" * 60)
    print(confusion_matrix.to_string())
    print("-" * 60)
    print("Legend: rows=actual labels, columns=predicted labels")
    print("=" * 60)

    y_true_binary = [1 if label == "vulnerable" else 0 for label in y_true]
    y_score = [1 if label == "vulnerable" else 0 for label in y_pred]
    if len(set(y_true_binary)) > 1:
        fpr, tpr, _ = roc_curve(y_true_binary, y_score)
        roc_auc = auc(fpr, tpr)
        print(f"ROC AUC: {roc_auc:.3f}")
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
        plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        plt.show()
    else:
        print("ROC AUC: unavailable (only one class in ground truth).")

    if report_data["mode"] == "paired":
        before_completed = df_results[f"{metric_prefix}_completed_before"].mean() if f"{metric_prefix}_completed_before" in df_results.columns and total_rows else 0
        after_completed = df_results[f"{metric_prefix}_completed_after"].mean() if f"{metric_prefix}_completed_after" in df_results.columns and total_rows else 0
        print(f"Completion rate (before-code): {before_completed:.2%}")
        print(f"Completion rate (after-code): {after_completed:.2%}")

        avg_before_time = df_results[f"{metric_prefix}_scan_time_before_seconds"].mean() if f"{metric_prefix}_scan_time_before_seconds" in df_results.columns and total_rows else 0
        avg_after_time = df_results[f"{metric_prefix}_scan_time_after_seconds"].mean() if f"{metric_prefix}_scan_time_after_seconds" in df_results.columns and total_rows else 0
        total_before_time = df_results[f"{metric_prefix}_scan_time_before_seconds"].sum() if f"{metric_prefix}_scan_time_before_seconds" in df_results.columns and total_rows else 0
        total_after_time = df_results[f"{metric_prefix}_scan_time_after_seconds"].sum() if f"{metric_prefix}_scan_time_after_seconds" in df_results.columns and total_rows else 0
        total_time = total_before_time + total_after_time

        print(f"\nTiming Statistics:")
        print(f"Before-code - Total: {total_before_time:.3f}s | Average: {avg_before_time:.3f}s")
        print(f"After-code  - Total: {total_after_time:.3f}s | Average: {avg_after_time:.3f}s")
        print(f"Total scan time: {total_time:.3f}s")
    else:
        completed_rate = df_results[f"{metric_prefix}_completed"].mean() if f"{metric_prefix}_completed" in df_results.columns and total_rows else 0
        print(f"Completion rate: {completed_rate:.2%}")
        total_time = df_results[f"{metric_prefix}_scan_time_seconds"].sum() if f"{metric_prefix}_scan_time_seconds" in df_results.columns and total_rows else 0
        avg_time = df_results[f"{metric_prefix}_scan_time_seconds"].mean() if f"{metric_prefix}_scan_time_seconds" in df_results.columns and total_rows else 0
        print(f"\nTiming Statistics:")
        print(f"Total: {total_time:.3f}s | Average: {avg_time:.3f}s")

    models_used = sorted(df_results[f"{metric_prefix}_model"].dropna().unique().tolist()) if f"{metric_prefix}_model" in df_results.columns else []
    if models_used:
        print(f"\nModel(s) in results: {', '.join(models_used)}")
    print("=" * 60)

In [ ]:
# Load and analyze benchmark results from a saved CSV file
from sklearn.metrics import classification_report, roc_curve, auc
import matplotlib.pyplot as plt

def analyze_benchmark_csv(csv_path):
    """
    Load a previously generated benchmark CSV and calculate all metrics.

    Supports both paired benchmark outputs and single-sample benchmark outputs.
    """
    try:
        df = pd.read_csv(csv_path)

        print("=" * 60)
        print(f"ANALYZING BENCHMARK CSV: {csv_path}")
        print("=" * 60)

        total_rows = len(df)
        print(f"Total rows/pairs analyzed: {total_rows}")

        if any(c.startswith("llm_") for c in df.columns):
            metric_prefix = "llm"
        elif any(c.startswith("bandit_") for c in df.columns):
            metric_prefix = "bandit"
        else:
            metric_prefix = None

        if metric_prefix is None:
            print("No supported metric columns found (expected llm_* or bandit_*).")
            return

        report_data = _build_benchmark_report_data(df, metric_prefix)
        if report_data is None:
            print("No supported classification columns found.")
            return

        y_true = report_data["y_true"]
        y_pred = report_data["y_pred"]
        total_predictions = report_data["total_predictions"]

        if report_data["mode"] == "paired":
            completed_before_col = f"{metric_prefix}_completed_before"
            completed_after_col = f"{metric_prefix}_completed_after"
            if completed_before_col in df.columns and completed_after_col in df.columns:
                failed_before = int((df[completed_before_col] != True).sum()) if total_rows else 0
                failed_after = int((df[completed_after_col] != True).sum()) if total_rows else 0
                tests_failed = failed_before + failed_after
                print(f"Unsuccessful tests: {tests_failed}/{total_predictions}")
        else:
            completed_col = f"{metric_prefix}_completed"
            if completed_col in df.columns:
                tests_failed = int((df[completed_col] != True).sum()) if total_rows else 0
                print(f"Tests failed: {tests_failed}/{total_predictions}")

        print("\n" + "-" * 60)
        print("Classification Report (total predictions)")
        print("-" * 60)
        print(classification_report(y_true, y_pred, labels=["vulnerable", "patched"], target_names=["vulnerable", "patched"], zero_division=0, digits=3))

        pred_series = pd.Series(y_pred, name="Predicted")
        pred_series = pred_series.where(pred_series.isin(["vulnerable", "patched"]), "unknown")
        actual_series = pd.Series(y_true, name="Actual")

        confusion_matrix = pd.crosstab(actual_series, pred_series, dropna=False)
        display_cols = ["vulnerable", "patched", "unknown"]
        present_cols = [c for c in display_cols if c in confusion_matrix.columns]
        confusion_matrix = confusion_matrix.reindex(index=["vulnerable", "patched"], columns=present_cols, fill_value=0)
        confusion_matrix.index = [f"Actual: {idx}" for idx in confusion_matrix.index]
        confusion_matrix.columns = [f"Predicted: {col}" for col in confusion_matrix.columns]

        print("-" * 60)
        print("Confusion Matrix (total predictions)")
        print("-" * 60)
        print(confusion_matrix.to_string())
        print("-" * 60)
        print("Legend: rows=actual labels, columns=predicted labels")
        print("=" * 60)

        y_true_binary = [1 if label == "vulnerable" else 0 for label in y_true]
        y_score = [1 if label == "vulnerable" else 0 for label in y_pred]
        if len(set(y_true_binary)) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)
            print(f"ROC AUC: {roc_auc:.3f}")
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
            plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("ROC Curve")
            plt.legend(loc="lower right")
            plt.grid(alpha=0.3)
            plt.show()
        else:
            print("ROC AUC: unavailable (only one class in ground truth).")

        if report_data["mode"] == "paired":
            before_completed_col = f"{metric_prefix}_completed_before"
            after_completed_col = f"{metric_prefix}_completed_after"
            before_completed = df[before_completed_col].mean() if before_completed_col in df.columns and total_rows else 0
            after_completed = df[after_completed_col].mean() if after_completed_col in df.columns and total_rows else 0
            print(f"Completion rate (before-code): {before_completed:.2%}")
            print(f"Completion rate (after-code): {after_completed:.2%}")

            before_time_col = f"{metric_prefix}_scan_time_before_seconds"
            after_time_col = f"{metric_prefix}_scan_time_after_seconds"
            avg_before_time = df[before_time_col].mean() if before_time_col in df.columns and total_rows else 0
            avg_after_time = df[after_time_col].mean() if after_time_col in df.columns and total_rows else 0
            total_before_time = df[before_time_col].sum() if before_time_col in df.columns and total_rows else 0
            total_after_time = df[after_time_col].sum() if after_time_col in df.columns and total_rows else 0
            total_time = total_before_time + total_after_time
            print(f"\nTiming Statistics:")
            print(f"Before-code - Total: {total_before_time:.3f}s | Average: {avg_before_time:.3f}s")
            print(f"After-code  - Total: {total_after_time:.3f}s | Average: {avg_after_time:.3f}s")
            print(f"Total scan time: {total_time:.3f}s")
        else:
            completed_col = f"{metric_prefix}_completed"
            time_col = f"{metric_prefix}_scan_time_seconds"
            if completed_col in df.columns:
                completion_rate = df[completed_col].mean() if total_rows else 0
                print(f"Completion rate: {completion_rate:.2%}")
            if time_col in df.columns:
                avg_time = df[time_col].mean() if total_rows else 0
                total_time = df[time_col].sum() if total_rows else 0
                print(f"\nTiming Statistics:")
                print(f"Total: {total_time:.3f}s | Average: {avg_time:.3f}s")

        model_col = f"{metric_prefix}_model"
        if model_col in df.columns:
            models_used = sorted(df[model_col].dropna().unique().tolist())
            if models_used:
                print(f"\nModel(s) in results: {', '.join(models_used)}")

        print("=" * 60)

    except FileNotFoundError:
        print(f"Error: Could not find file at {csv_path}")
    except Exception as e:
        print(f"Error reading CSV: {e}")

input_csv_path = "splitData.csv"

# Check if file exists before running
import os
if os.path.exists(input_csv_path):
    analyze_benchmark_csv(input_csv_path)
else:
    print(f"File not found: {input_csv_path}")
    print("Available CSV files in results/:")
    if os.path.exists("results"):
        for f in os.listdir("results"):
            if f.endswith(".csv"):
                print(f"  - {f}")

In [ ]:
# Debug Bandit Input/Output from CSV rows
def debug_bandit_from_csv(
    csv_path,
    sample_pairs=1,
    start_row=0,
    random_sample=False,
    random_seed=42,
    before_col="vulnerable_code",
    after_col="fixed_code",
    code_col=None,
    label_col=None,
):
    df = pd.read_csv(csv_path)
    if df.empty:
        print("CSV is empty.")
        return

    schema = _detect_benchmark_input_schema(
        df,
        before_col=before_col,
        after_col=after_col,
        code_col=code_col,
        label_col=label_col,
    )

    total_rows = len(df)
    sample_pairs = int(sample_pairs) if sample_pairs is not None else 1
    if sample_pairs <= 0:
        print("sample_pairs must be >= 1")
        return

    if random_sample:
        sample_n = min(sample_pairs, total_rows)
        sampled_df = df.sample(n=sample_n, random_state=random_seed)
        selected_indices = sampled_df.index.tolist()
    else:
        if start_row < 0 or start_row >= total_rows:
            print(f"start_row {start_row} is out of range (0-{total_rows-1})")
            return
        end_row = min(start_row + sample_pairs, total_rows)
        selected_indices = list(range(start_row, end_row))

    print("=" * 60)
    if schema["mode"] == "paired":
        print(f"BANDIT CSV DEBUG | Testing {len(selected_indices)} pair(s)")
        print(f"Columns: before={schema['before_col']}, after={schema['after_col']}")
    else:
        print(f"BANDIT CSV DEBUG | Testing {len(selected_indices)} sample(s)")
        print(f"Columns: code={schema['code_col']}, label={schema['label_col']}")
    print("=" * 60)

    for i, row_index in enumerate(selected_indices, start=1):
        row = df.iloc[row_index]
        print("-" * 60)
        print(f"Sample {i}/{len(selected_indices)} | Row {row_index + 1}")

        if schema["mode"] == "paired":
            before_code = str(row.get(schema["before_col"], "") or "")
            after_code = str(row.get(schema["after_col"], "") or "")

            print(f"Source lengths: before={len(before_code)} chars, after={len(after_code)} chars")
            print(f"Line counts: before={before_code.count(chr(10)) + 1 if before_code else 0}, after={after_code.count(chr(10)) + 1 if after_code else 0}")
            print(f"Before/after identical: {before_code == after_code}")

            b_completed, b_pred, _, b_time, b_raw, b_prompt = test_vulnerabilities_benchmark(before_code, include_raw_response=True, include_prompt=True)
            a_completed, a_pred, _, a_time, a_raw, a_prompt = test_vulnerabilities_benchmark(after_code, include_raw_response=True, include_prompt=True)
            b_label = label_from_prediction(b_completed, b_pred)
            a_label = label_from_prediction(a_completed, a_pred)
            b_success = b_label == "vulnerable"
            a_success = a_label == "safe"

            print(f"Before preview:\n{before_code[:240]}")
            if len(before_code) > 240:
                print("... [before preview truncated]")
            print(f"After preview:\n{after_code[:240]}")
            if len(after_code) > 240:
                print("... [after preview truncated]")

            print("\n" + "=" * 60)
            print("FULL PROMPT RECEIVED BY BANDIT (BEFORE CODE):")
            print("=" * 60)
            print(b_prompt if b_prompt else "[Prompt unavailable]")
            print("=" * 60)
            print(f"Before expected: vulnerable | predicted: {b_label} | success: {b_success} | completed: {b_completed} | time: {b_time:.3f}s")
            print(f"Before raw response: {repr(b_raw)}")

            print("\n" + "=" * 60)
            print("FULL PROMPT RECEIVED BY BANDIT (AFTER CODE):")
            print("=" * 60)
            print(a_prompt if a_prompt else "[Prompt unavailable]")
            print("=" * 60)
            print(f"After expected: safe       | predicted: {a_label} | success: {a_success} | completed: {a_completed} | time: {a_time:.3f}s")
            print(f"After raw response: {repr(a_raw)}")
        else:
            code_text = str(row.get(schema["code_col"], "") or "")
            expected_label = _normalize_benchmark_label(row.get(schema["label_col"], ""))
            print(f"Source length: code={len(code_text)} chars")
            print(f"Line count: {code_text.count(chr(10)) + 1 if code_text else 0}")
            print(f"Code preview:\n{code_text[:240]}")
            if len(code_text) > 240:
                print("... [code preview truncated]")

            completed, pred, _, scan_time = test_vulnerabilities_benchmark(code_text)
            predicted_label = label_from_prediction(completed, pred)
            success = predicted_label == expected_label

            print(f"Expected: {expected_label} | predicted: {predicted_label} | success: {success} | completed: {completed} | time: {scan_time:.3f}s")
            print(f"Last Bandit raw output (first 1000 chars):")
            print(last_bandit_output[:1000])

# Example debug usage
input_csv_path = "splitData.csv"
# debug_bandit_from_csv(input_csv_path, sample_pairs=3, start_row=90)
debug_bandit_from_csv(input_csv_path, sample_pairs=5, random_sample=True, random_seed=42)